# Lab 1: Evaluating Your Agent with Built-in Metrics

This notebook demonstrates how to evaluate the deployed **fixFirstAgent** using Amazon Bedrock AgentCore's built-in evaluators. You'll learn to measure agent quality across multiple dimensions — helpfulness, correctness, and relevance — using two complementary approaches.

### Why Evaluate?

Deploying an agent is only the beginning. Without measurement, you can't know:
- Is the agent actually helping customers solve their problems?
- Are its responses factually accurate?
- Does it stay on-topic or drift into irrelevant answers?

AgentCore Evaluations uses **LLM-as-a-Judge** — a separate LLM scores your agent's responses against predefined criteria. This gives you quantitative metrics without needing human reviewers.

### Two Evaluation Approaches

| | Batch Evaluation | Online Evaluation |
|---|---|---|
| **When it runs** | On-demand, when you trigger it | Continuously in the background |
| **What it evaluates** | All sessions in a time window | Every session as it completes |
| **How you get results** | Poll `get-batch-evaluation` API | Check CloudWatch Logs/Metrics |
| **Use case** | Testing before release, CI/CD | Production monitoring |
| **Latency** | Minutes (depends on session count) | ~15 min after session idle timeout |

### Architecture

<pre>
                        FixFirst Agent (with OTel instrumentation)
                                        |
                                        | OTel spans
                                        v
                    +-------------------------------------------+
                    |     CloudWatch Log Group                   |
                    |  /aws/bedrock-agentcore/runtimes/          |
                    |         {runtimeId}-DEFAULT                |
                    +-------------------+-----------------------+
                                        |
                     +------------------+------------------+
                     |                                     |
                     v                                     v
    +-------------------------------+     +-------------------------------+
    |      Batch Evaluation         |     |      Online Evaluation        |
    |  (start-batch-evaluation)     |     |   (CfnOnlineEvaluationConfig) |
    |                               |     |                               |
    |  Discovers sessions from      |     |  Monitors log group           |
    |  log group, scores them all   |     |  continuously, scores each    |
    |  with selected evaluators     |     |  session after 15 min idle    |
    +-------------------------------+     +-------------------------------+
                     |                                     |
                     v                                     v
    +-------------------------------+     +-------------------------------+
    |  get-batch-evaluation API     |     |  CloudWatch Logs + Metrics    |
    |  -> evaluatorSummaries:       |     |  /aws/bedrock-agentcore/      |
    |     avgScore per evaluator    |     |  evaluations/results/{id}     |
    +-------------------------------+     +-------------------------------+
</pre>

### Available Built-in Evaluators

| Evaluator | Level | Category | What it measures |
|---|---|---|---|
| `Builtin.Helpfulness` | Trace | Response Quality | How useful the response is to the user |
| `Builtin.Correctness` | Trace | Response Quality | Factual accuracy of the information |
| `Builtin.ResponseRelevance` | Trace | Response Quality | Whether the response addresses the query |
| `Builtin.Faithfulness` | Trace | Response Quality | Whether response is supported by context |
| `Builtin.Conciseness` | Trace | Response Quality | Appropriate brevity without missing info |
| `Builtin.Coherence` | Trace | Response Quality | Logical structure |
| `Builtin.InstructionFollowing` | Trace | Response Quality | How well agent follows system instructions |
| `Builtin.GoalSuccessRate` | Session | Task Completion | Whether the agent achieved the user's goal |
| `Builtin.ToolSelectionAccuracy` | Tool call | Component | Whether the right tool was selected |
| `Builtin.ToolParameterAccuracy` | Tool call | Component | Whether tool parameters were extracted correctly |
| `Builtin.Harmfulness` | Trace | Safety | Whether response contains harmful content |
| `Builtin.Stereotyping` | Trace | Safety | Whether content makes group generalizations |

This lab uses **Helpfulness**, **Correctness**, and **ResponseRelevance** — the most appropriate metrics for a single-turn appliance troubleshooting agent.

📚 [Evaluators](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluators.html) | [Batch Evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/batch-evaluations-start.html) | [Online Evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/create-online-evaluations.html)

**Prerequisites:**
- fixFirstAgent deployed with OTel instrumentation (`opentelemetry-instrument` entrypoint)
- Node.js (for CDK)
- AWS CLI >= 2.34
- `bash_kernel` (`pip install bash_kernel && python -m bash_kernel.install`)

**Note:** This notebook uses the **Bash kernel**. Select *Kernel → Change Kernel → Bash* if not already selected.

**Estimated time:** ~15 minutes (batch eval takes 5-10 min to complete)

## 1. Setup

Configure environment variables and verify the agent is deployed and ready.

The agent uses **Cognito JWT authentication** — you need a test user to invoke it. If you haven't created one yet:

```bash
POOL_ID=$(aws ssm get-parameter --name /fixFirstAgent/cognito-user-pool-id --query Parameter.Value --output text --region $REGION)
aws cognito-idp admin-create-user --user-pool-id $POOL_ID --username evaluser --message-action SUPPRESS --region $REGION
aws cognito-idp admin-set-user-password --user-pool-id $POOL_ID --username evaluser --password 'EvalTest1!' --permanent --region $REGION
```

> Update `HOME_DIR` below if your repository is cloned to a different location.

In [ ]:
export HOME_DIR=~/sample-aws-agentops-fix-first-agent
export REGION=${AWS_REGION:-$(aws configure get region 2>/dev/null)}
export REGION=${REGION:-us-east-1}
export COGNITO_USERNAME=evaluser
export COGNITO_PASSWORD='EvalTest1!'

# Check prerequisites
$HOME_DIR/ab_testing/scripts/check_prerequisites.sh

# Get runtime info
RUNTIME_ARN=$(aws ssm get-parameter --name /fixFirstAgent/agentcore-runtime-arn --query Parameter.Value --output text --region $REGION)
RUNTIME_ID=$(echo "$RUNTIME_ARN" | awk -F/ '{print $NF}')
CLIENT_ID=$(aws ssm get-parameter --name /fixFirstAgent/cognito-client-id --query Parameter.Value --output text --region $REGION)
LOG_GROUP="/aws/bedrock-agentcore/runtimes/${RUNTIME_ID}-DEFAULT"

STATUS=$(aws bedrock-agentcore-control get-agent-runtime --agent-runtime-id "$RUNTIME_ID" --region $REGION --query status --output text)
echo "Runtime:   $RUNTIME_ID  Status: $STATUS"
echo "Log group: $LOG_GROUP"

## 2. List Available Evaluators

Query the AgentCore control plane to see all built-in evaluators available in your account. Each evaluator has a **type** (Builtin vs Custom) and a **level** (Trace, Session, or Tool call) that determines what it scores.

In [ ]:
aws bedrock-agentcore-control list-evaluators --max-results 20 --region $REGION \
    --output table --query 'evaluators[].{Name:evaluatorName,Type:evaluatorType,Level:level}'

## 3. Deploy Online Evaluation Config

Deploy a separate CDK stack (`evaluations/cdk/`) that sets up continuous background evaluation for your agent.

**What gets created:**
- An **IAM role** for the evaluation service (permissions to read CloudWatch Logs and invoke the LLM judge)
- An **Online Evaluation Config** pointing to your runtime's log group

**How it works:**
- The config monitors `/aws/bedrock-agentcore/runtimes/{runtimeId}-DEFAULT`
- Every session is scored automatically after 15 min of inactivity
- Three evaluators run on each session: **Helpfulness**, **Correctness**, **ResponseRelevance**
- Results are published to CloudWatch Metrics (namespace `Bedrock-AgentCore/Evaluations`)

The CDK stack reads the runtime ARN from SSM — no hardcoded values.

📚 [Create Online Evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/create-online-evaluations.html)

In [ ]:
cd $HOME_DIR/ab_testing/evaluations/cdk
npm install
npx cdk deploy fixFirstAgent-EvaluationsStack --require-approval never

## 4. Send Traffic

Send 15 appliance troubleshooting prompts to the agent using `invoke_agent.py`.

**What the script does:**
1. Reads prompts from `ab_testing/prompts.txt` (20 real-world appliance problems)
2. Authenticates with Cognito (`USER_PASSWORD_AUTH`) to get a JWT access token
3. For each prompt, generates a unique session ID and sends an HTTP POST to the runtime's invoke URL with the Bearer token
4. Prints each response snippet

The runtime (with OTel instrumentation) automatically emits spans to CloudWatch for each invocation. Both the batch evaluation and the online evaluation will use these spans.

📚 [Invoke an AgentCore Runtime agent](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-invoke-agent.html)

In [ ]:
cd $HOME_DIR/ab_testing/evaluations
python3 invoke_agent.py --region $REGION --count 15

## 5. Batch Evaluation

Start a batch evaluation job that discovers all sessions in the runtime's log group and scores them with three built-in evaluators.

**How batch evaluation works:**
1. You call `start-batch-evaluation` with the log group, service name, and evaluator list
2. The service discovers all sessions in the log group (or a filtered subset)
3. For each session, it downloads the OTel spans and passes them to each evaluator (LLM judge)
4. Results are aggregated — average score per evaluator across all sessions
5. You poll `get-batch-evaluation` until status is `COMPLETED`

**Evaluators used:**
- `Builtin.Helpfulness` — Is the response useful to someone with an appliance problem?
- `Builtin.Correctness` — Is the troubleshooting advice factually accurate?
- `Builtin.ResponseRelevance` — Does the response actually address the specific appliance issue asked about?

> **Note:** Wait ~2 minutes after sending traffic for spans to appear in CloudWatch before running this cell.

📚 [Start Batch Evaluation](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/batch-evaluations-start.html) | [Understanding Results](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/batch-evaluations-results.html)

In [ ]:
EVAL_NAME="fixFirstAgent_batch_$(date +%s)"

RESPONSE=$(aws bedrock-agentcore start-batch-evaluation \
    --batch-evaluation-name "$EVAL_NAME" \
    --evaluators '[{"evaluatorId":"Builtin.Helpfulness"},{"evaluatorId":"Builtin.Correctness"},{"evaluatorId":"Builtin.ResponseRelevance"}]' \
    --data-source-config '{"cloudWatchLogs":{"serviceNames":["fixFirstAgent_Agent.DEFAULT"],"logGroupNames":["'$LOG_GROUP'"]}}' \
    --region $REGION --output json)

BATCH_ID=$(echo $RESPONSE | python3 -c "import sys,json;print(json.load(sys.stdin)['batchEvaluationId'])")
echo "Batch evaluation started: $BATCH_ID"
echo "Polling for results (this may take 5-10 minutes)..."
echo

while true; do
    RESULT=$(aws bedrock-agentcore get-batch-evaluation --batch-evaluation-id "$BATCH_ID" --region $REGION --output json)
    STATUS=$(echo $RESULT | python3 -c "import sys,json;print(json.load(sys.stdin)['status'])")
    echo "  Status: $STATUS"
    if [[ "$STATUS" == "COMPLETED" || "$STATUS" == "COMPLETED_WITH_ERRORS" || "$STATUS" == "FAILED" || "$STATUS" == "STOPPED" ]]; then
        break
    fi
    sleep 30
done

echo
echo "=== Results ==="
aws bedrock-agentcore get-batch-evaluation --batch-evaluation-id "$BATCH_ID" --region $REGION \
    --output table \
    --query 'evaluationResults.{Sessions:numberOfSessionsCompleted,Summaries:evaluatorSummaries[].{Evaluator:evaluatorId,AvgScore:statistics.averageScore,Evaluated:totalEvaluated,Failed:totalFailed}}'

## 6. Online Evaluation Results

The online evaluator (deployed in step 3) runs in the background and scores each session after ~15 minutes of inactivity. Results are written to a dedicated CloudWatch log group.

**Re-run this cell after ~20 minutes** to see online evaluation scores. If the log group doesn't exist yet, it means no sessions have been scored — wait longer.

📚 [Results and Output](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/results-and-output.html)

In [ ]:
EVAL_CONFIG_ID=$(aws ssm get-parameter --name /fixFirstAgent/online-eval-config-id --query Parameter.Value --output text --region $REGION)
LOG_GROUP_RESULTS="/aws/bedrock-agentcore/evaluations/results/$EVAL_CONFIG_ID"

echo "Config: $EVAL_CONFIG_ID"
echo "Log group: $LOG_GROUP_RESULTS"
echo

QUERY_ID=$(aws logs start-query --log-group-name "$LOG_GROUP_RESULTS" \
    --start-time $(date -d '60 minutes ago' +%s 2>/dev/null || date -v-60M +%s) \
    --end-time $(date +%s) \
    --query-string 'fields @timestamp, @message | sort @timestamp desc | limit 20' \
    --region $REGION --output text --query queryId 2>/dev/null)

if [ -z "$QUERY_ID" ]; then
    echo "Log group does not exist yet. Sessions are scored ~15 min after idle timeout."
    echo "Re-run this cell later."
else
    sleep 5
    aws logs get-query-results --query-id "$QUERY_ID" --region $REGION --output table
fi

## 7. Cleanup

Destroy the evaluations CDK stack. This removes the online evaluation config and its IAM role. The agent itself is not affected — it continues running.

In [ ]:
cd $HOME_DIR/ab_testing/evaluations/cdk
npx cdk destroy fixFirstAgent-EvaluationsStack --force

## Interpreting Results

### Score Scale

All built-in evaluators use LLM-as-a-Judge scoring on a 0–1 scale:

| Score | Meaning | Action |
|-------|---------|--------|
| 0.83+ | Excellent ("Very Helpful", "Correct") | Agent is performing well |
| 0.50–0.82 | Adequate ("Somewhat Helpful") | Room for improvement |
| < 0.50 | Poor ("Not Helpful", "Incorrect") | Needs prompt/model changes |

### What Low Scores Mean

| Metric | Low score indicates | Suggested fix |
|--------|--------------------|--------------|
| Helpfulness | Responses aren't actionable | Better system prompt (Lab 2) |
| Correctness | Factual errors in responses | More capable model (Lab 3) |
| ResponseRelevance | Agent drifts off-topic | Tighter system prompt, add guardrails |

### Batch vs Online Results

- **Batch** gives you aggregate averages across all sessions — good for comparing before/after changes
- **Online** gives per-session scores continuously — good for catching regressions in production

### Next Steps

Once you have baseline scores, use A/B testing to measure improvements:
- **Lab 2** — Configuration bundle A/B test: same model, different system prompt
- **Lab 3** — Target-based A/B test: different models (Nova Lite vs Claude)

📚 [Results & Output](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/results-and-output.html) | [A/B Testing](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/ab-testing.html)